<a href="https://colab.research.google.com/github/rchougule12995/Air-Quality-Index-India/blob/master/performance_facebook_commission_small_scale_sampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ***This notebook include all necessary steps for executing lightweight AB test sample selection by analysts;***

#Part I: Setting Up the Library imports, inputs and target KPIs

## Step 1: Import Libraries

In [ ]:
import copy
import pickle
import pprint
import random
from collections import defaultdict
from datetime import datetime
from datetime import timedelta
from itertools import groupby
from types import SimpleNamespace
import statsmodels.formula.api as smf

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats
from scipy import stats
from scipy.stats import ttest_rel
from toolz import partial
from scipy.optimize import fmin_slsqp

#Authenticator
from google.colab import auth
auth.authenticate_user()

# Convenience.
infinite_defaultdict = lambda: defaultdict(infinite_defaultdict)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

# Plotting.
#%matplotlib inline
#plt.style.use('seaborn')

pp = pprint.PrettyPrinter()

# Linking to BQ
from google.cloud import bigquery

client = bigquery.Client(project='sojern-datascience-dev')
auth.authenticate_user()

import threading

import ipywidgets as widgets
from IPython.display import display, clear_output

pd.options.display.max_columns = None
pd.options.display.max_rows = None


## Step 2 (Option 1): Pull Data from BQ

Table Usage:

Meta and SEM (Custom Queries)

In [ ]:
# Pulling all Eligible campaigns based on analyst's selection cretierias
# Data can be pulled directly using BQ codes in Colab or from a BQ table created in BQ by analysts
# load data from BQ

%%bigquery eligible_list --project sojern-bigquery

WITH eligible_camps AS (
  SELECT
    opp_id
    ,opp_name
    ,service_tier
    ,region
    ,SUM(li_imps) AS li_imps
    ,SUM(li_media_cost) AS li_media_cost
    ,SUM(li_conversions) AS li_conversions
    ,SUM(li_rev) AS li_rev
    ,SUM(order_conversions) AS order_conversions
    ,SUM(pixel_fires) AS pixel_fires
    ,SUM(NR) AS NR
  FROM
    `sojern-operations-analytics.emerging_facebook.JZ_high_level_test_platform_level_reseller_detail` a
  WHERE 1=1
  AND CAST(a.date AS date) >= current_date() - 30
  AND delivery_platform = 'facebook'
  AND opp_id IN (
  '0065b00000t0y2lAAA',	'006VQ00000AaGjdYAF',	'006VQ00000Fc5R3YAJ',	'006VQ00000BP5arYAD',	'006VQ00000CKe2PYAT',	'006VQ00000Dyc69YAB',	'006VQ00000CKNCrYAP',	'006VQ000009jgyoYAA',	'006VQ00000A4sX9YAJ',	'006VQ000006nFFPYA2',	'006VQ00000EU4AzYAL',	'0065b000014OK5KAAW',	'006VQ00000Ch13bYAB',	'006VQ00000E3x3VYAR',	'006VQ00000Cx2YLYAZ',	'006VQ00000A01xeYAB',	'006VQ00000CPwWXYA1',	'006VQ00000BpStcYAF',	'006VQ00000CXDNlYAP',	'006VQ00000BpVORYA3',	'006VQ00000AmGW9YAN',	'006VQ00000BEmHlYAL',	'0065b0000125kQ3AAI',	'0065b000010acvpAAA',	'006VQ00000EeY4YYAV',	'006VQ000008y22NYAQ',	'006VQ00000EAQkLYAX',	'006VQ00000AoRLsYAN',	'006VQ00000CX0TlYAL',	'006VQ00000CWxalYAD',	'006VQ00000ArL6zYAF',	'006VQ000009epjNYAQ',	'006VQ000006b0FJYAY',	'006VQ00000AXbSrYAL',	'006VQ000009eqltYAA',	'006VQ00000G01irYAB',	'0065b00000pqmViAAI',	'006VQ000007cB01YAE',	'006VQ00000CWqpOYAT',	'006VQ00000BEmWHYA1',	'006VQ00000BdRE0YAN',	'0063900000qg4o9AAA',	'006VQ000009z75pYAA',	'006VQ000009ej7iYAA',	'006VQ000008k28fYAA',	'0065b000014OIRpAAO',	'006VQ00000ETtsHYAT',	'006VQ00000BwUVdYAN',	'006VQ000002LNzmYAG',	'006VQ00000BdiFlYAJ',	'006VQ000008k81BYAQ',	'006VQ000003Ych4YAC',	'006VQ000004HBchYAG',	'006VQ00000CKdKtYAL',	'006VQ000003BHK0YAO',	'006VQ000004vTB3YAM',	'006VQ00000CWwd3YAD',	'006VQ000004GliDYAS',	'006VQ00000BBnLdYAL',	'006VQ00000EVsdbYAD',	'006VQ000009zxR0YAI',	'006VQ000003CzKXYA0',	'006VQ00000ESMK6YAP',	'006VQ000005LtWWYA0',	'006VQ00000A02QfYAJ',	'006VQ00000Ch99BYAR',	'006VQ00000EDAW9YAP',	'006VQ000009hSYeYAM',	'006VQ00000CX0TmYAL',	'006VQ000009erEvYAI',	'006VQ00000CWxxMYAT',	'006VQ00000CWtP7YAL',	'006VQ00000DBj2NYAT',	'006VQ00000DnYaYYAV',	'006VQ00000BdfGHYAZ',	'006VQ00000EBiTtYAL',	'0065b00000yqc6QAAQ',	'006VQ00000BdiXWYAZ',	'006VQ000004EHRiYAO',	'006VQ000009eqYzYAI',	'006VQ000003s175YAA',	'006VQ00000CX0jtYAD',	'006VQ00000A01cfYAB',	'006VQ00000G06vRYAR',	'006VQ00000E6Qm9YAF',	'006VQ0000083vgHYAQ',	'006VQ000008Lyz3YAC',	'006VQ00000CX20vYAD',	'006VQ000005nCunYAE',	'006VQ000009V85lYAC',	'006VQ00000CWvgzYAD',	'006VQ00000Bf0vOYAR',	'006VQ00000BpKnzYAF',	'006VQ00000FMrQ1YAL',	'006VQ000009lmwzYAA',	'006VQ00000F9TJvYAN',	'006VQ000009WTbBYAW',	'006VQ000001vJgVYAU',	'006VQ00000G09gPYAR',	'006VQ0000029baHYAQ',	'0065b0000124nuxAAA',	'006VQ00000BW4RxYAL',	'006VQ00000CWsmNYAT',	'006VQ000002IL3IYAW',	'006VQ00000Bp4o2YAB',	'006VQ00000CWyjiYAD',	'006VQ00000CX0aEYAT',	'006VQ00000BpWfRYAV',	'006VQ000009qgh9YAA',	'006VQ00000G08HJYAZ',	'006VQ000005NM8EYAW',	'006VQ00000CWv7VYAT',	'006VQ000008tQOjYAM',	'006VQ000005jyDFYAY',	'006VQ000009hSYZYA2',	'006VQ00000BfDEHYA3',	'006VQ00000Cty9VYAR',	'006VQ00000CfUuUYAV',	'006VQ00000CX1PpYAL',	'006VQ00000CWuMmYAL',	'006VQ00000BdNWHYA3',	'006VQ000002NoCjYAK',	'006VQ00000CklHxYAJ',	'006VQ00000CVDACYA5',	'006VQ000008gAKwYAM',	'006VQ000009hSYcYAM',	'0065b0000127IgnAAE',	'006VQ000009X3MvYAK',	'006VQ000008u97FYAQ',	'006VQ00000F61s2YAB',	'006VQ00000EAB0ZYAX',	'006VQ00000B8TZaYAN',	'0065b0000124dqTAAQ',	'006VQ000009zz9TYAQ',	'0065b0000124eRzAAI',	'006VQ000007pHZ3YAM',	'006VQ00000CX22XYAT',	'0065b0000128QL2AAM',	'006VQ000009Fh2gYAC',	'0065b000014OnS5AAK',	'006VQ00000F6vRCYAZ',	'006VQ00000CWvykYAD',	'006VQ000006ouJhYAI',	'006VQ00000DwbCeYAJ',	'006VQ000008V6xNYAS',	'006VQ00000756tZYAQ',	'006VQ000002hmoJYAQ',	'0065b0000133PabAAE',	'006VQ000009hA2AYAU',	'006VQ000005tYnmYAE',	'0065b000014NnhPAAS',	'0065b00000sOZu0AAG',	'006VQ000003CPSNYA4',	'006VQ00000FO1n7YAD',	'0063900000kq2X7AAI',	'0065b000014NbRZAA0',	'006VQ000003NaGXYA0',	'006VQ00000BGluIYAT',	'006VQ00000CkdQsYAJ',	'006VQ000005cy3VYAQ',	'006VQ000001erHyYAI',	'006VQ000006YNQDYA4',	'006VQ00000G1SXJYA3',	'0065b0000131OaAAAU',	'006VQ000006QzlzYAC',	'006VQ000009RyB1YAK',	'006VQ000003c5zSYAQ',	'006VQ00000DsMgnYAF',	'006VQ000001vOEkYAM',	'006VQ000008fuMbYAI',	'006VQ000001Dp7PYAS',	'0067000000k4FSHAA2',	'006VQ00000CkhizYAB',	'006VQ00000A7mQMYAZ',	'0065b000010dKaUAAU',	'006VQ000005ueRMYAY',	'006VQ000006QmSCYA0',	'006VQ000006PntJYAS',	'006VQ000006ovJ3YAI',	'0065b0000133ucYAAQ',	'0065b000014NcRqAAK',	'006VQ000004EJWjYAO',	'006VQ000003dW9aYAE',	'006VQ000006y17RYAQ',	'0065b000014NhT5AAK',	'006VQ000005tYz5YAE',	'006VQ00000FMjnTYAT',	'006VQ000008gAw2YAE',	'006VQ0000054yU9YAI',	'006VQ000002NoCiYAK',	'006VQ000006ovJ2YAI',	'006VQ000005HtrgYAC',	'006VQ00000Bf5tRYAR',	'006VQ000006R3ckYAC',	'006VQ000007fUcPYAU',	'0063p00000xJKczAAG',	'006VQ00000DrK17YAF',	'006VQ00000Bl7pJYAR',	'006VQ00000BGA9xYAH',	'006VQ00000Bdo3RYAR',	'006VQ0000059374YAA',	'006VQ000008g4vUYAQ',	'006VQ00000BfDZFYA3',	'006VQ00000593ztYAA',	'006VQ000009hSYbYAM',	'006VQ000007UOQfYAO',	'006VQ000002E1UXYA0',	'006VQ000009lmwsYAA',	'006VQ00000G067SYAR',	'006VQ000009lmwyYAA',	'006VQ000008gGVBYA2',	'006VQ000005qr8cYAA',	'006VQ00000BfCzlYAF',	'006VQ000009hSYfYAM',	'0065b0000134eHtAAI',	'006VQ00000BdiR3YAJ',	'006VQ00000Ckh7tYAB',	'006VQ000009uXaQYAU',	'006VQ000006N69JYAS',	'006VQ00000BdOnJYAV',	'006VQ000008OCHxYAO',	'006VQ000003pmk1YAA',	'006VQ00000G03UXYAZ',	'006VQ000009wKKYYA2',	'006VQ00000FzyY7YAJ',	'006VQ000007zrEfYAI',	'006VQ00000BqRqbYAF',	'006VQ000009ucWrYAI',	'006VQ00000DsOaXYAV',	'006VQ000006MIqrYAG',	'006VQ0000072amXYAQ',	'006VQ00000G02jmYAB',	'006VQ000009ucbhYAA',	'006VQ00000BfDcTYAV',	'006VQ00000BdhYGYAZ',	'006VQ00000ENNDZYA5',	'006VQ00000Fc670YAB',	'006VQ000005JgdOYAS',	'006VQ000009hSYdYAM',	'006VQ000007YuVBYA0',	'0065b0000134gclAAA',	'006VQ000006A9pmYAC',	'006VQ0000072Yw2YAE',	'006VQ00000Bf5jlYAB',	'006VQ00000BdL7uYAF',	'0065b000014AguRAAS',	'006VQ00000Eza05YAB',	'006VQ000003ZZCTYA4',	'006VQ00000BMuoQYAT',	'0065b0000127FmJAAU',	'006VQ00000EmyBVYAZ',	'006VQ000004evlFYAQ',	'006VQ000009ViKYYA0',	'0063900000uPiw0AAC',	'0065b0000125sMPAAY',	'0065b0000126cJtAAI',	'006VQ000002MGWQYA4',	'006VQ00000DvmLNYAZ',	'006VQ000003ZJnfYAG',	'006VQ00000Bdj5NYAR',	'0065b0000127bXIAAY',	'006VQ00000EedLxYAJ',	'006VQ00000CPw1tYAD',	'006VQ000001SHYPYA4',	'006VQ00000ArIx7YAF',	'0065b000014ORnDAAW',	'006VQ00000CkhUTYAZ',	'0065b000014AUDhAAO',	'006VQ00000F12juYAB',	'006VQ00000CzCVCYA3',	'006VQ00000Bf16gYAB',	'006VQ0000083Ze9YAE',	'006VQ00000Bf1JZYAZ',	'0065b000014NMG3AAO',	'006VQ000009mA37YAE',	'006VQ0000046JIfYAM',	'006VQ000004lZo1YAE',	'0063p00000xJOQKAA4',	'006VQ00000BeE18YAF',	'0065b000010dJ57AAE',	'0065b0000125D1nAAE',	'006VQ00000CIvddYAD',	'006VQ00000GPsGwYAL',	'006VQ000008KPAXYA4',	'006VQ000002cAJoYAM',	'006VQ000005JgdTYAS',	'006VQ00000Bf16fYAB',	'0065b00000vsru3AAA',	'0065b00000vqJHnAAM',	'0065b0000126ByWAAU',	'006VQ000005uRkEYAU',	'0065b0000125seJAAQ',	'006VQ000005qR4UYAU',	'006VQ000007d9llYAA',	'006VQ000007Op0YYAS',	'006VQ000009hSYaYAM',	'006VQ00000CFYwvYAH',	'006VQ00000AUq0EYAT',	'006VQ000005wmW1YAI',	'006VQ00000CkjOEYAZ',	'0065b00001354lnAAA',	'006VQ00000Bg16DYAR',	'0063p00000uGWn1AAG',	'006VQ000008Syz7YAC',	'0065b000014AO63AAG',	'006VQ00000BEMf3YAH',	'006VQ000005JgdWYAS',	'006VQ00000BdpPJYAZ',	'006VQ000007hNJtYAM',	'006VQ00000BdQNhYAN',	'006VQ00000AqCk9YAF',	'006VQ000005TeSDYA0',	'0065b000012bkuDAAQ',	'006VQ00000F12LhYAJ',	'006VQ00000BdM7DYAV',	'006VQ000009KSscYAG',	'006VQ00000BdcrvYAB',	'006VQ00000BdOk5YAF',	'006VQ000005LePOYA0',	'006VQ000009lmwuYAA',	'006VQ0000087oDCYAY',	'006VQ00000FFAODYA5',	'006VQ00000DvmlBYAR',	'006VQ00000AUNWWYA5',	'006VQ00000BdMjtYAF',	'006VQ00000CkfXWYAZ',	'006VQ00000BdFKDYA3',	'006VQ00000CkhuHYAR',	'006VQ00000BWZIrYAP',	'006VQ000006YYYVYA4',	'006VQ000004EGU1YAO',	'0065b000010cSOEAA2',	'006VQ000009TZd7YAG',	'006VQ0000071WJZYA2',	'006VQ0000044kwVYAQ',	'006VQ00000CMcaoYAD',	'006VQ000003pk5WYAQ',	'006VQ000006TEM9YAO',	'006VQ000006Qpt3YAC',	'006VQ00000BfDPZYA3',	'0063900000pjk83AAA',	'006VQ000003ZWeQYAW',	'0065b00001357neAAA',	'006VQ00000F0yDGYAZ',	'006VQ00000Gn4pGYAR',	'0065b00000t1PPgAAM',	'006VQ000009hSYYYA2',	'006VQ00000Gime2YAB',	'0065b0000125AmyAAE',	'006VQ000003r5nrYAA',	'006VQ000009sigPYAQ',	'006VQ00000EzPFxYAN',	'0063p00000y6fspAAA',	'006VQ000009KBzpYAG',	'006VQ00000Bf0s9YAB',	'006VQ00000A01HiYAJ',	'006VQ000002wTWjYAM',	'006VQ000003iLv3YAE',	'006VQ00000CFoK9YAL',	'006VQ000007kMpXYAU',	'0063p00000y4DqNAAU',	'0065b0000134FtRAAU',	'006VQ000008h90DYAQ',	'006VQ00000Bf0wzYAB',	'006VQ0000089RFhYAM',	'006VQ00000FkPsRYAV',	'006VQ00000BdLFyYAN',	'006VQ00000CP6HqYAL',	'0065b0000126Pb4AAE',	'006VQ00000EqjVeYAJ',	'006VQ00000GnDuAYAV',	'0065b00000ysM27AAE',	'0063p00000xJTobAAG',	'006VQ00000ECBR3YAP',	'006VQ000008h90MYAQ',	'006VQ000009uJ4WYAU',	'0065b0000126Jo0AAE',	'006VQ00000D9hSrYAJ',	'0065b00000vqro1AAA',	'006VQ00000GQ5KbYAL',	'006VQ0000038apNYAQ',	'006VQ000008D0LlYAK',	'006VQ00000Bbwd2YAB',	'0063p00000xbjHVAAY',	'0063900000uQHAcAAO',	'006VQ00000FqHXBYA3',	'006VQ000009uJ4XYAU',	'0065b000014OLFTAA4',	'006VQ00000BdLUTYA3',	'006VQ0000032GBqYAM',	'006VQ00000CwUYJYA3',	'006VQ000003ZRutYAG',	'006VQ00000Ckhe9YAB',	'006VQ00000GQyybYAD',	'006VQ00000BIhTCYA1',	'006VQ00000E0uyEYAR',	'0065b000014NGKuAAO',	'006VQ00000BezuUYAR',	'006VQ000007Nh53YAC',	'006VQ000007hHUbYAM',	'006VQ00000Bf0LuYAJ',	'006VQ0000032IvEYAU',	'006VQ00000BdJNqYAN',	'006VQ000005I7kwYAC',	'006VQ000004C8IsYAK',	'006VQ000003zdp7YAA',	'006VQ00000GAlEAYA1',	'006VQ00000CkgTaYAJ',	'006VQ0000083eyoYAA',	'006VQ000001zDPZYA2',	'0065b00000uFegjAAC',	'006VQ0000083e7cYAA',	'006VQ000008VfQTYA0',	'006VQ000001XkZlYAK',	'006VQ000008GuHlYAK',	'006VQ00000FA9CfYAL',	'006VQ00000FlbczYAB',	'006VQ00000595ATYAY',	'006VQ00000EI57dYAD',	'0065b0000124viNAAQ',	'006VQ000005JgdRYAS',	'006VQ00000DlyhBYAR',	'006VQ000007fUFpYAM',	'006VQ000008dV8zYAE',	'006VQ000002N37KYAS',	'006VQ00000BwV8LYAV',	'0065b00001255YzAAI',	'006VQ00000AJ5CnYAL',	'006VQ00000CPLK5YAP',	'0065b000014NefeAAC',	'0065b0000101GmvAAE',	'006VQ000003wD4nYAE',	'006VQ000004gQ0dYAE',	'006VQ00000EzZVYYA3',	'006VQ000009aAJqYAM',	'006VQ00000Cr77iYAB',	'006VQ00000C4ucbYAB',	'006VQ00000BdRA5YAN',	'006VQ00000ExwccYAB',	'006VQ000009ItDlYAK',	'006VQ00000F825dYAB',	'006VQ00000CcDmvYAF',	'0065b0000132jkTAAQ',	'006VQ000001yEsQYAU',	'006VQ00000CPviXYAT',	'006VQ000002Pck6YAC',	'006VQ00000G076kYAB',	'006VQ00000G7xtOYAR',	'0065b000014OHwFAAW',	'006VQ00000Fo9QrYAJ',	'006VQ00000Dp73DYAR',	'006VQ000007APxBYAW',	'006VQ0000097PR8YAM',	'006VQ000006DHytYAG',	'006VQ00000CkhFxYAJ',	'006VQ00000Bdm4sYAB',	'006VQ00000BdPxtYAF',	'006VQ000004r9eSYAQ',	'006VQ000004Z0ndYAC',	'006VQ000009ZMq5YAG',	'006VQ000001vOhlYAE',	'006VQ000009V5EMYA0',	'006VQ000006HyUBYA0',	'006VQ000009aFZdYAM',	'006VQ000002v7ejYAA',	'006VQ000003LcgUYAS',	'0065b000012bFVNAA2',	'006VQ00000BNKVyYAP',	'0065b00000sQTTBAA4',	'006VQ00000418fZYAQ',	'006VQ00000EeoIzYAJ',	'006VQ00000AupgPYAR',	'006VQ000005nBCLYA2',	'006VQ00000GSInEYAX',	'006VQ00000CMdexYAD',	'006VQ00000EmRInYAN',	'006VQ000005JgdVYAS',	'006VQ000006isvkYAA',	'006VQ000005tRXlYAM',	'006VQ000005wXyPYAU',	'006VQ00000ArEn5YAF',	'0065b00001258PuAAI',	'006VQ00000Fl7n0YAB',	'006VQ000007kKPVYA2',	'006VQ000007s1EQYAY',	'006VQ000003ZTQRYA4',	'006VQ000008h90PYAQ',	'006VQ00000GlR3aYAF',	'006VQ00000DhYgUYAV',	'006VQ000007EgL7YAK',	'006VQ00000GJ04kYAD',	'006VQ000008h90IYAQ',	'006VQ00000F6rKLYAZ',	'006VQ00000Ez9SzYAJ',	'006VQ000009HfcfYAC',	'006VQ000008uCl3YAE',	'006VQ0000055KCvYAM',	'006VQ000009jYefYAE',	'006VQ00000BDZssYAH',	'006VQ000009GtD3YAK',	'006VQ00000Fbq0bYAB',	'006VQ00000F7SIHYA3',	'006VQ000009s9HGYAY',	'006VQ00000CUuttYAD',	'006VQ00000EH7oDYAT',	'006VQ0000072EfZYAU',	'006VQ00000Am9rFYAR',	'006VQ000006nCFxYAM',	'006VQ00000GlNzHYAV',	'006VQ00000BTyosYAD',	'006VQ00000C2OCDYA3',	'0065b0000127XzSAAU',	'006VQ000008uthxYAA',	'006VQ000005jCN8YAM',	'006VQ000005UCQfYAO',	'006VQ000006MHOXYA4',	'006VQ00000860f4YAA',	'006VQ00000F0zVtYAJ',	'006VQ00000Dn8hhYAB',	'006VQ000009SlmnYAC',	'006VQ0000087RlFYAU',	'0065b000014NdzUAAS',	'006VQ00000DSaYvYAL',	'006VQ000008h90GYAQ',	'0063p00000xJTovAAG',	'006VQ000003iFBLYA2',	'006VQ00000Gh8IsYAJ',	'0065b000014OTBqAAO',	'006VQ00000CooAoYAJ',	'006VQ000001E4rBYAS',	'006VQ00000EedAfYAJ',	'006VQ00000FaZEkYAN',	'006VQ000003uy2zYAA',	'006VQ000003weEiYAI',	'006VQ000007u8WjYAI',	'0065b00001285PyAAI',	'006VQ00000CyuurYAB',	'006VQ000006DFvRYAW',	'006VQ00000FeLt7YAF',	'006VQ00000GCgofYAD',	'006VQ000004cxPnYAI',	'006VQ000007fXwrYAE',	'006VQ00000BNr6xYAD',	'006VQ00000EWCXRYA5',	'006VQ000007fXlZYAU',	'006VQ000007jJC9YAM',	'006VQ00000DSMR8YAP',	'0065b0000125yAzAAI',	'006VQ000007fUpJYAU',	'006VQ000008hjgPYAQ',	'0065b000014OIZKAA4',	'006VQ00000BBAXIYA5',	'006VQ000007fPxhYAE',	'006VQ00000FsqIjYAJ',	'006VQ000008rFAnYAM',	'006VQ00000592VyYAI',	'006VQ0000038vXHYAY',	'006VQ00000AKHq9YAH',	'006VQ00000BlDJiYAN',	'006VQ00000BpS3xYAF',	'006VQ00000ByhWtYAJ',	'006VQ00000DP2C2YAL',	'006VQ000008h90LYAQ',	'006VQ000007fOlVYAU',	'006VQ00000FoVEUYA3',	'006VQ0000092xYHYAY',	'006VQ00000CHrLxYAL',	'006VQ000007fXDhYAM',	'0065b0000125OoJAAU',	'006VQ000009eoXCYAY',	'006VQ000005Gx67YAC',	'006VQ00000FJoi1YAD',	'006VQ00000B8woXYAR',	'006VQ00000B8xmAYAR',	'006VQ00000DM3vqYAD',	'006VQ00000EzddtYAB',	'006VQ00000BGttSYAT',	'006VQ000005pOv3YAE',	'006VQ000004J1PVYA0',	'006VQ00000DSZrNYAX',	'0065b0000134NnlAAE',	'006VQ000003TLfFYAW',	'0065b0000133uXdAAI',	'006VQ000009qiz3YAA',	'0065b00000zyEedAAE',	'006VQ000002rJ1mYAE',	'006VQ00000DOt78YAD',	'006VQ00000Cwk5BYAR',	'006VQ000007fSU9YAM',	'006VQ00000DSPsNYAX',	'006VQ000007dqU9YAI',	'006VQ00000EzToFYAV',	'006VQ00000G5Oy9YAF',	'006VQ00000Fcz9tYAB',	'006VQ000009GWy2YAG',	'006VQ00000AJ5CoYAL',	'006VQ00000GlGo6YAF',	'006VQ00000BEIMvYAP',	'006VQ00000CqyCTYAZ',	'006VQ000009jM7FYAU',	'006VQ00000EtEYZYA3',	'006VQ00000DNO5NYAX',	'006VQ000006C5F0YAK',	'006VQ000009RtBJYA0',	'006VQ000008h90EYAQ',	'006VQ000005QkKcYAK',	'0065b00001258LJAAY',	'006VQ00000G5RBFYA3',	'006VQ000005tQ3qYAE',	'006VQ00000GYBflYAH',	'006VQ00000EudJZYAZ',	'0065b0000134hn0AAA',	'006VQ00000DP1mEYAT',	'006VQ0000051vJ7YAI',	'006VQ00000HKApSYAX',	'006VQ000009SF8bYAG',	'006VQ00000FE4ntYAD',	'006VQ00000GoJhPYAV',	'0063p00000uGeVkAAK',	'006VQ000005ECrGYAW',	'006VQ00000EzVtHYAV',	'006VQ0000057mbLYAQ',	'0063p00000y4DKPAA2',	'006VQ00000DSWofYAH',	'006VQ00000HJbOfYAL',	'006VQ00000Bf5ztYAB',	'006VQ00000BGKE9YAP',	'006VQ00000BlBjFYAV',	'0065b0000132mu7AAA',	'006VQ000004Z2JCYA0',	'006VQ000007dzQzYAI',	'006VQ00000HAXLuYAP',	'006VQ00000CHRo5YAH',	'006VQ00000HK6XJYA1',	'006VQ000005c2HFYAY',	'006VQ00000I3Ji5YAF',	'006VQ00000Hcfl7YAB',	'0065b000014NnrOAAS',	'006VQ00000Fq6ezYAB',	'0065b00000sP078AAC',	'006VQ00000GnuvvYAB',	'006VQ000005h0BZYAY',	'006VQ00000Gwfj7YAB',	'006VQ000004dZEvYAM',	'006VQ00000FnejdYAB',	'006VQ000006D4QZYA0',	'006VQ00000EfDajYAF',	'006VQ00000BTezqYAD',	'006VQ00000Bo2TJYAZ',	'006VQ00000HSEZJYA5',	'006VQ000009I1oTYAS',	'006VQ000008uEzlYAE',	'006VQ00000CYWo9YAH',	'0065b000014AUDDAA4',	'006VQ000001jGUfYAM',	'006VQ000006A1nNYAS',	'006VQ000007QlqTYAS',	'006VQ00000BQuQ2YAL',	'006VQ000005KdvCYAS',	'006VQ0000092yR7YAI',	'006VQ000007UTF4YAO'
      )
  GROUP BY ALL
      )
SELECT *
FROM eligible_camps

In [ ]:
## Accept an input from the Analyst team member which would allow them to specify the KPI's they're looking for
## If there are any numeric columns in the table, by default we can set it to be a float
## A unique identifier can be converted to a category type, accept the unique identifier from the analyst team member

## Segment based on the percentile of the data, allow the analyst team members to define the percentile of the data to be selected and included in the file.

## Threshold of acceptance, allow multiple ranges? (1 to 5)

In [ ]:
eligible_list=eligible_list

In [ ]:
eligible_list.head()

## Step 2 (Option 2): Import Data from a csv File

In [ ]:
# # Import data from local drive
# #import from local drive
# from google.colab import files
# uploaded = files.upload()
# `
# import io
# eligible_list = pd.read_csv(io.BytesIO(uploaded['List for Small AB Test_Eligible.csv']))

In [ ]:
# # viewing data file
# eligible_list.head(5)


##Step 3: Identifying key variables

### Selecting the Campaign Identifier and KPIs

In [ ]:
# Convert numeric columns to float, keep others as string
for col in eligible_list.columns:
  if pd.api.types.is_numeric_dtype(eligible_list[col]):
    eligible_list[col] = eligible_list[col].astype(float)
  else:
    eligible_list[col] = eligible_list[col].astype(str)

eligible_list.info()

In [ ]:
sample = 10    #This is the number of samples you want to draw for test group. The same number will be drawn for control groups.
threshold_of_acceptance_kpis = [1,2,3,4,5]

options = list(eligible_list.columns)
item_layout = widgets.Layout(margin='0 0 10px 0')

# Layout for the VBox containing each dropdown and its description
item_layout = widgets.Layout(
    margin='0 0 20px 0'  # Adds 20px of space below each item
)

# Layout for the dropdowns and text areas to ensure they have a fixed width
widget_layout = widgets.Layout(width='420px')

# --- Campaign Identifier ---
campaign_identifier_dropdown = widgets.Dropdown(
    options=options,
    value=None,
    placeholder='Select Campaign Identifier',
    description='Campaign Identifier:',
    layout=widget_layout,
    style={'description_width': 'initial'} # Ensures the description text is not cut off
)
campaign_identifier_description = widgets.Text(
    value='',
    placeholder='Select the field for Campaign Identifier below',
    layout=widget_layout
)
campaign_identifier_box = widgets.VBox([campaign_identifier_description, campaign_identifier_dropdown])
# --- KPI 1 ---
kpi1_dropdown = widgets.Dropdown(
    options=options,
    value=None,
    placeholder='Select a KPI',
    description='KPI 1:',
    layout=widget_layout,
    style={'description_width': 'initial'}
)
kpi1_description = widgets.Text(
    value='',
    placeholder='Select the field for the primary KPI below',
    layout=widget_layout
)
kpi1_box = widgets.VBox([kpi1_description, kpi1_dropdown])

# --- KPI 2 ---
kpi2_dropdown = widgets.Dropdown(
    options=options,
    value=None,
    placeholder='Select a KPI',
    description='KPI 2:',
    layout=widget_layout,
    style={'description_width': 'initial'}
)
kpi2_description = widgets.Text(
    value='',
    placeholder='Select the field for secondary KPI below or leave blank if not needed',
    layout=widget_layout
)
kpi2_box = widgets.VBox([kpi2_description, kpi2_dropdown])


# --- KPI 3 ---
kpi3_dropdown = widgets.Dropdown(
    options=options,
    value=None,
    placeholder='Select a KPI',
    description='KPI 3:',
    layout=widget_layout,
    style={'description_width': 'initial'}
)
kpi3_description = widgets.Text(
    value='',
    placeholder='Select the field for secondary KPI below or leave blank if not needed',
    layout=widget_layout
)
kpi3_box = widgets.VBox([kpi3_description, kpi3_dropdown])

# Vertically arrange all the components with spacing
all_kpis = widgets.VBox(
    [
        widgets.Box([campaign_identifier_box], layout=item_layout),
        widgets.Box([kpi1_box], layout=item_layout),
        widgets.Box([kpi2_box], layout=item_layout),
        widgets.Box([kpi3_box], layout=item_layout)
    ]
)

display(all_kpis)

#Part II: Having made selection for the Campaign Identifier now Run all the cells below at once using (Command/Control + F10)

In [ ]:
kpis=[kpi1_dropdown.value,kpi2_dropdown.value,kpi3_dropdown.value]
kpis = list(filter(lambda x: x is not None, kpis))

In [ ]:
kpis

In [ ]:
campaign_identifier=campaign_identifier_dropdown.value

In [ ]:
print(campaign_identifier)

## Filtering out the low performers

In [ ]:
for kpi in kpis:
  print("\nSummary Statistic: Distribution of the KPI: "+kpi)
  print(eligible_list[kpi].describe())

In [ ]:
#Selecting specific Quantile for each of the KPIs
for kpi in kpis:
  q_low = eligible_list[kpi].quantile(0.03)
  q_high  = eligible_list[kpi].quantile(0.97)
  eligible_list = eligible_list[(eligible_list[kpi] < q_high) & (eligible_list[kpi] >= q_low)]

In [ ]:
for kpi in kpis:
  print("\nSummary Statistic: Distribution of the KPI: "+kpi)
  print(eligible_list[kpi].describe())

**These values need to be filled by the Analyst for the rest of the code to run**

In [ ]:
eligible_list[campaign_identifier] = eligible_list[campaign_identifier].dropna()

In [ ]:
for i in kpis:
  eligible_list[i] = eligible_list[i].fillna(0)

In [ ]:
# Select columns needed for sampling
eligible_list = eligible_list.groupby([campaign_identifier])[kpis].sum().reset_index()
eligible_list.head()

## Step 4: Randomization

In [ ]:
# Count of unique campaigns
eligible_list[campaign_identifier].nunique()

## Obtain Test Group

In [ ]:
# Obtain Test Group
eligible_list_test = eligible_list[campaign_identifier].sample(n=sample, random_state=400)
eligible_list_test = eligible_list_test.to_list()

In [ ]:
eligible_list_test

In [ ]:
#Ensuring exactly the same number of samples were picked as required
assert len(eligible_list_test) == sample

### Test Group Mean and Standard Deviation

In [ ]:
test_group = eligible_list[eligible_list[campaign_identifier].isin(eligible_list_test)]
kpi_mean = {}
kpi_variance = {}
for kpi in kpis:
  result_kpi=scipy.stats.describe(test_group[kpi], ddof=1, bias=False)
  print("\nSummary Statistics/Distribution of the KPI: "+kpi)
  print(test_group[kpi].describe().reset_index().to_string(index=False, header=False))
  kpi_mean[kpi]=result_kpi[2]
  kpi_variance[kpi]=result_kpi[3]
  print('KPI mean:'  , kpi_mean[kpi])
  print('KPI Variance:'  , kpi_variance[kpi])

In [ ]:
test_group[campaign_identifier].nunique()

### Obtain Control Group

In [ ]:
# Left over campaign IDs for selecting the control group (removing the test group)
eligible_list_control = eligible_list[~eligible_list[campaign_identifier].isin(eligible_list_test)]

In [ ]:
def find_control_group(threshold, result_list):
    np.random.seed(threshold) # Use threshold as seed for demonstration; in a real scenario, use a different seed or generate uniquely for each thread.
    for i in range(5000):  # Reduce the range for faster execution
        np.random.seed(i + threshold * 10000) # Combine iteration and threshold for seed
        list_control = eligible_list_control[campaign_identifier].sample(n=sample)
        control_group = eligible_list[eligible_list[campaign_identifier].isin(list_control)]

        kpi_means_control = {}
        kpi_var_control = {}
        for kpi in kpis:
            kpi_means_control[kpi] = control_group[kpi].mean()
            kpi_var_control[kpi] = control_group[kpi].var()

        # Calculate percentage differences for all KPIs
        percentage_diffs_mean = {}
        percentage_diffs_var = {}
        all_thresholds_met = True

        for idx, kpi in enumerate(kpis):
            threshold_value = threshold_of_acceptance_kpis[idx] if idx < len(threshold_of_acceptance_kpis) else 1 # Use provided thresholds or default to 1
            test_mean = kpi_mean[kpi]
            test_var = kpi_variance[kpi]
            if test_mean != 0:
                percentage_diffs_mean[kpi] = abs((kpi_means_control[kpi] - test_mean) * 100 / test_mean)
                percentage_diffs_var[kpi] = abs((kpi_var_control[kpi] - test_var) * 100 / test_var)
                if percentage_diffs_mean[kpi] > threshold_value:
                    all_thresholds_met = False
                    break
            else:
                 if kpi_means_control[kpi] != 0:
                     all_thresholds_met = False
                     break


        if all_thresholds_met:
            result_list.append({'seed': i + threshold * 10000, kpis[0] + '_mean_diff': percentage_diffs_mean[kpis[0]]})
            result_list.append({'seed': i + threshold * 10000, kpis[0] + '_var_diff': percentage_diffs_var[kpis[0]]})


results = []
threads = []

# Create and start a thread for each threshold value
for threshold in threshold_of_acceptance_kpis:
    thread = threading.Thread(target=find_control_group, args=(threshold, results))
    threads.append(thread)
    thread.start()

# Wait for all threads to complete
for thread in threads:
    thread.join()

# Find the result with the minimum mean difference for the first KPI
if results:
    best_sample = min(results, key=lambda x: x.get(kpis[0] + '_mean_diff', float('inf')))
    best_seed = best_sample.get("seed")
    filtered_results = [result for result in results if result.get('seed') == best_seed]
    print("\nBest control sample found:")
    print(f"Seed: {best_seed}")
    print(f"Mean difference for {kpis[0]}: {filtered_results[0][kpis[0] + '_mean_diff']:.2f}%")
    print(f"Variance difference for {kpis[0]}: {filtered_results[1][kpis[0] + '_var_diff']:.2f}%")

    # We can now use the seed from best_sample to regenerate the control group
    best_seed = best_sample['seed']
    np.random.seed(best_seed)
    final_control_list = eligible_list_control[campaign_identifier].sample(n=sample)
    control_group = eligible_list[eligible_list[campaign_identifier].isin(final_control_list)]

    print("\nFinal Control Group Summary Statistics:")
    for kpi in kpis:
        print(f"\nKPI: {kpi}")
        print(control_group[kpi].describe().reset_index().to_string(index=False, header=False))

else:
    print("\nNo control group found that meets the acceptance thresholds.")

In [ ]:
# def find_control_group_weighted(thresholds, max_iterations=10000):
#     """
#     Finds a control group that minimizes a weighted combination of percentage mean
#     and absolute standard deviation differences from the test group for multiple KPIs.

#     Args:
#         thresholds (list): List of percentage mean difference thresholds for each KPI.
#                            The order corresponds to the weighting (first element highest weight).
#         max_iterations (int): The maximum number of random samples to check.

#     Returns:
#         tuple: A tuple containing the best control group DataFrame found and a dictionary
#                with the differences (mean and std) for the best sample, or None and
#                an empty dictionary if no suitable sample is found within the iterations.
#     """
#     eligible_control_ids = eligible_list_control[campaign_identifier]
#     best_control_group = None
#     min_weighted_diff = float('inf')
#     best_diffs = {}

#     # Calculate the mean and std dev for the test group for the relevant KPIs
#     test_kpi_stats = {}
#     for kpi in kpis:
#         test_kpi_stats[kpi] = {
#             'mean': test_group[kpi].mean(),
#             'std': test_group[kpi].std() # Using std for absolute difference calculation
#         }

#     for i in range(max_iterations):
#         # Sample a control group
#         try:
#             current_control_ids = eligible_control_ids.sample(n=sample, random_state=i)
#         except ValueError:
#             print(f"Warning: Could not sample {sample} items from control group. Stopping search early at iteration {i}.")
#             break # Not enough control campaigns left to sample

#         current_control_group = eligible_list[eligible_list[campaign_identifier].isin(current_control_ids)]

#         current_diffs = {}
#         weighted_diff = 0
#         all_mean_thresholds_met = True

#         for idx, kpi in enumerate(kpis):
#             test_mean = test_kpi_stats[kpi]['mean']
#             test_std = test_kpi_stats[kpi]['std']
#             control_mean = current_control_group[kpi].mean()
#             control_std = current_control_group[kpi].std()

#             # Calculate percentage mean difference
#             if test_mean != 0:
#                 mean_diff_percent = abs((control_mean - test_mean) * 100 / test_mean)
#             else:
#                 mean_diff_percent = abs(control_mean) * 100 # If test mean is 0, difference is absolute control mean

#             # Calculate absolute standard deviation difference
#             std_diff_abs = abs(control_std - test_std)

#             current_diffs[f'{kpi}_mean_diff_percent'] = mean_diff_percent
#             current_diffs[f'{kpi}_std_diff_abs'] = std_diff_abs

#             # Check if the mean difference meets the threshold for this KPI
#             if idx < len(thresholds):
#                 if mean_diff_percent > thresholds[idx]:
#                     all_mean_thresholds_met = False
#                     break # No need to check other KPIs for this sample if one fails the mean threshold

#             # Add weighted difference to total
#             # Higher weight for earlier KPIs (lower index)
#             weight = len(kpis) - idx
#             weighted_diff += mean_diff_percent * weight
#             weighted_diff += std_diff_abs * weight # Also weight the std diff

#         # If all mean thresholds are met, check if this sample is better based on weighted difference
#         if all_mean_thresholds_met:
#             if weighted_diff < min_weighted_diff:
#                 min_weighted_diff = weighted_diff
#                 best_control_group = current_control_group
#                 best_diffs = current_diffs # Store the differences for the best sample

#     return best_control_group, best_diffs

# # --- Run the weighted control group selection ---
# threshold_of_acceptance_kpis_percent = threshold_of_acceptance_kpis # Ensure this is in percentage points

# # print(f"Searching for best control group among {max_iterations} random samples...")
# best_control_group_df, best_diffs = find_control_group_weighted(threshold_of_acceptance_kpis_percent)

# if best_control_group_df is not None:
#     print("\nBest Control Group Found:")
#     print(best_control_group_df[campaign_identifier].to_list())

#     print("\nDifferences with Test Group for Best Control Sample:")
#     for kpi in kpis:
#         mean_diff = best_diffs.get(f'{kpi}_mean_diff_percent')
#         std_diff = best_diffs.get(f'{kpi}_std_diff_abs')
#         print(f"  KPI: {kpi}")
#         print(f"    % Mean Difference: {mean_diff:.2f}%")
#         print(f"    Absolute Std Difference: {std_diff:.2f}") # Print absolute std difference

#     print("\nBest Control Group Summary Statistics:")
#     for kpi in kpis:
#         print(f"\nKPI: {kpi}")
#         print(best_control_group_df[kpi].describe().reset_index().to_string(index=False, header=False))

# else:
#     print("\nNo control group found that meets the mean difference acceptance thresholds within the given iterations.")
#     print("Consider increasing max_iterations or loosening the thresholds.")


In [ ]:
control_group.describe()

In [ ]:
test_group.describe()

In [ ]:
#Checking to ensure same number of campaigns in test and control group
assert control_group[campaign_identifier].nunique() == test_group[campaign_identifier].nunique()

In [ ]:
control_group[campaign_identifier].unique()

## Step 5: Prebalance Check

In [ ]:
for kpi in kpis:
  print("\nSummary Statistic: Distribution of Test Group for the KPI: "+kpi)
  test_group = eligible_list[eligible_list[campaign_identifier].isin(eligible_list_test)]
  result = scipy.stats.describe(test_group[kpi], ddof=1, bias=False)
  print(result)

In [ ]:
for kpi in kpis:
  print("\nSummary Statistic: Distribution of Control Group for the KPI: "+kpi)
  control_group = eligible_list[eligible_list[campaign_identifier].isin(control_group.opp_id.to_list())]
  result = scipy.stats.describe(control_group[kpi], ddof=1, bias=False)
  print(result)

## Step 6: Exporting Final List of Test Group and Control Group

In [ ]:
# Ensuring to see the two groups are not exactly the same
a = set(test_group[campaign_identifier].to_list())
b = set(control_group[campaign_identifier].to_list())
if a == b:
    print("Test and control are the same")
else:
    print("Test and control are not the same")


In [ ]:
# Ensuring that each element in list 1 is not in list 2
list1 = test_group[campaign_identifier].to_list()
list2 = control_group[campaign_identifier].to_list()
if [item for item in list1 if item in list2]:
  print("At least one element common between Test and Control Group")
else:
  print("No element common between Test and Control Group")

In [ ]:
#Exporting Test Group as a csv
#test_group[campaign_identifier].to_csv("Test.csv")

In [ ]:
#Exporting Control Group as a csv
#control_group[campaign_identifier].to_csv("Control.csv")

In [ ]:
test = test_group[campaign_identifier].to_list()
test

In [ ]:
control = control_group[campaign_identifier].to_list()
control

In [ ]:
test_df = eligible_list[eligible_list[campaign_identifier].isin(test)]
test_df.to_csv("test_group.csv")

In [ ]:
control_df = eligible_list[eligible_list[campaign_identifier].isin(control)]
control_df.to_csv("control_group.csv")